# Notebook for experiments with skeletonization

Imports

In [ ]:
import sys
import os

# Go up three levels from the notebook's directory (../src) to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..')) 
# Add the src directory to the path
src_path = os.path.join(project_root, 'src') 
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    
        
print(f"Project Root: {project_root}")
print(f"Src Path added: {src_path}")
print(f"sys.path[0]: {sys.path[0]}") 

In [ ]:
from typing import List, Tuple, Optional
import numpy as np
import networkx as nx
import os
from cv2 import imread
from ypstruct import structure
import matplotlib.pyplot as plt
import numpy as np


from skeletonization.converter import Converter
from skeletonization.network_simplification import NetworkSimplification
from skeletonization.skeleton_gng_mapper import SkeletonGNGMapper
from skeletonization.settings import Settings, gng_parameters

Additional parameters:

In [ ]:
epsilon = 6
binary_threshold = 160

Iterate over images in folder function

In [ ]:
def iterate_over_images_in_folder(folder_path: str):
    for file in os.listdir(folder_path):
        if file.endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif")):
            yield file


In [ ]:
import importlib
import skeletonization.visualization as viz
importlib.reload(viz)

from skeletonization.visualization import (
    Vector,
    Point,
    GraphTraversal,
    find_top_leftmost_point,
    calculate_label_position,
    plot_network_result,
)

In [ ]:
def plot_graph_traversal_debug(
    ax: plt.Axes, G: nx.Graph, traversal: List[Tuple[Point, Optional[Vector]]]
):
    """Enhanced traversal plot with vector type labels and debug prints."""
    pos = {node: (G.nodes[node]["x"], G.nodes[node]["y"]) for node in G.nodes()}
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="lightgray", arrows=False)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=30, node_color="lightblue")

    for i, (point, vector) in enumerate(traversal):
        if i == 0:
            ax.scatter(point.x, point.y, color='red', s=100, zorder=10)
            
        if vector:
            print(
                f"Vector {i}: x1: {vector.x1}, y1: {vector.y1}, x2: {vector.x2}, y2: {vector.y2}"
            )
            dx = vector.x2 - vector.x1
            dy = vector.y2 - vector.y1
            abs_dx = abs(dx)
            abs_dy = abs(dy)
            
            if abs_dx == abs_dy:
                vector_type = "DiagonalVector"
            elif abs_dx > abs_dy:
                vector_type = "HorizontalVector"
            else:
                vector_type = "VerticalVector"

            ax.arrow(
                vector.x1, vector.y1, dx, dy,
                head_width=3, head_length=3,
                fc="r", ec="r",
                length_includes_head=True, alpha=0.5,
            )
            
            label_x, label_y = calculate_label_position(vector)
            ax.text(
                label_x, label_y, vector_type,
                ha="center", va="center",
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
            )

        ax.text(point.x, point.y, str(i), fontsize=8, ha="center", va="center")

    ax.set_title("Graph Traversal (Debug)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.axis("equal")
    ax.invert_yaxis()

In [ ]:
settings = Settings(
    kafka_topic="dummy",
    dlq_topic="dummy",
    kafka_bootstrap_servers="dummy",
    N=50,
    maxit=70,
    L=100,
    epsilon_b=0.2,
    epsilon_n=0.01,
    alpha=0.5,
    delta=0.995,
    T=50,
)

skeleton_gng_mapper = SkeletonGNGMapper(
    settings=settings, skeletonization_threshold=180, simplification_epsilon=5
)

# Get 10 random images from the folder
folder_path = "../../tests/prepared_samples/1_1"
# folder_path = '../../tests/generated_samples/mnist_1/train'
images = [
    os.path.join(folder_path, file)
    for file in sorted(os.listdir(folder_path))
    if file.endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif"))
]

class_number = 3
img_num = 15
image_id = f"mnist_{class_number}_{img_num:05d}.png"
local_path = f"../../tests/generated_samples/mnist_{class_number}/test"
# images = [os.path.join(local_path, image_id)]

for image_path in images:
    image = imread(image_path, 0)
    simplified_network, threshold = skeleton_gng_mapper.process_image(image)
    binary = skeleton_gng_mapper.binary_image(image, threshold)
    skeleton = skeleton_gng_mapper.skeletonize(image, threshold)
    image_points = skeleton_gng_mapper.skeleton_to_points(skeleton)
    network = skeleton_gng_mapper.fit_gng(image_points)
    simplified_network = NetworkSimplification.simplify_network(
        network, epsilon=skeleton_gng_mapper.simplification_epsilon
    )
    plot_network_result(
        image_path,
        network,
        gng_parameters(settings),
        image_points,
        image,
        binary,
        skeleton,
        simplified_network,
    )

    G = Converter.convert_simplified_network_to_networkx(simplified_network)

    cycles = list(nx.simple_cycles(G))
    print(f"Cycles: {len(cycles)}")
    if not nx.is_connected(G):
        print(f"Graph {image_path} is not connected")
        continue
    top_leftmost_point = find_top_leftmost_point(G)
    print(f"Top leftmost point: {G.nodes[top_leftmost_point]}")
    traversal = GraphTraversal(G).dfs_traversal(start_node=top_leftmost_point)

    plot_graph_traversal_debug(plt.gca(), G, traversal)
    plt.show()